## 다양한 작업에 대해 모델 비교하기

이 노트북에서는 Granite-7B-Instruct와 함께 또 다른 모델인 Flan-T5-Large를 병렬로 사용해보고, 그 성능을 비교해보겠습니다.

Flan-T5-Large는 우리가 기존에 사용하던 Granite-7B-Instruct 보다 작은 모델로, GPU 없이도 실행 가능하고 RAM 4GB만 사용합니다.
이 모델이 과연 주어진 작업을 잘 수행할 수 있을까요?

### 요구 사항 및 라이브러리 임포트

실습 지침에 따라 올바른 워크벤치 이미지를 선택하여 실행하였다면, 필요한 모든 라이브러리가 이미 설치되어 있을 것입니다.  
그렇지 않은 경우에는 다음 셀의 첫 번째 줄 주석을 해제하여 필요한 패키지를 설치하세요.

In [ ]:
# 아래 줄은 올바른 워크벤치 이미지를 선택하지 않았거나, 이 노트북을 워크숍 환경 외부에서 사용하는 경우에만 주석을 해제하십시오.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

import json
import time

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import HuggingFaceTextGenInference, VLLMOpenAI

### Langchain 파이프라인

이제 두 개의 서로 다른 LLM 엔드포인트와 두 개의 Langchain 파이프라인을 정의하겠습니다.

In [ ]:
# LLM Inference Server URL
inference_server_url = "http://granite-7b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLM definition
llm = VLLMOpenAI(           # 우리는 vLLM OpenAI 호환 API 클라이언트를 사용하고 있습니다. 하지만 모델은 OpenAI가 아니라 OpenShift AI에서 실행되고 있습니다.
    openai_api_key="EMPTY",   # 따라서 OpenAI 키가 필요하지 않습니다.
    openai_api_base= f"{inference_server_url}/v1",
    model_name="granite-7b-instruct",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# Flan-T5-Large LLM Inference Server URL
inference_server_url_flan_t5 = "http://llm-flant5.ic-shared-llm.svc.cluster.local:3000/"

# LLM definition
llm_flant5 = HuggingFaceTextGenInference(
    inference_server_url=inference_server_url_flan_t5,
    max_new_tokens=96,
    top_k=10,
    top_p=0.95,
    typical_p=0.95,
    temperature=0.01,
    repetition_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

두 모델은 모두 동일한 **템플릿**을 사용하게 됩니다.

In [ ]:
template="""<|system|>
You are a helpful, respectful and honest assistant.
Always assist with care, respect, and truth. Respond with utmost utility yet securely.
Avoid harmful, unethical, prejudiced, or negative content. Ensure replies promote fairness and positivity.
I will give you a text, then ask a question about it. Give a precise and as concise as possible answer to this question.
<|user|>
### TEXT:
{text}

### QUESTION:
{query}

### ANSWER:
<|assistant|>
"""
prompt = PromptTemplate(input_variables=["input"], template=template)

이제 2개의 모델에 질의할 수 있도록 2개의 **conversation** 객체를 생성할 수 있습니다.

In [ ]:
conversation = prompt | llm
conversation_flant5 = prompt | llm_flant5

이제 모델에 질의할 준비가 완료되었습니다!

이번 예제에서는 하나의 보험 청구에 대해서만 질의하고 결과를 확인해보겠습니다.  
물론, 원하신다면 다른 청구 내용으로도 자유롭게 실험해보셔도 됩니다.

In [ ]:
filename = 'claims/claim1.json'

# Opening JSON file
claims = {}
with open(filename, 'r') as file:
    data = json.load(file)
claims[filename] = data

# Content and queries
text_input = f"Subject: {claims[filename]['subject']}\nContent:\n{claims[filename]['content']}"
sentiment_query = "What is the sentiment of the person sending this claim?"
location_query = "Where does the event the claim is related to happen?"
time_query = "When does the event the claim is related to happen?"

# Analyze the claim
print(f"***************************")
print(f"* Claim: {filename}")
print(f"***************************")
print("Original content:")
print("-----------------")
print(f"Subject: {claims[filename]['subject']}\nContent:\n{claims[filename]['content']}\n\n")
print('Analysis with Granite-7B-Instruct:')
print("--------")
start_granite = time.time()
print(f"- Sentiment: ")
conversation.invoke(input={"text": text_input, "query": sentiment_query});
print("\n- Location: ")
conversation.invoke(input={"text": text_input, "query": location_query});
print("\n- Time: ")
conversation.invoke(input={"text": text_input, "query": time_query});
print("\n\n                          ----====----\n")
end_granite = time.time()
print('Analysis with Flan-T5-Large:')
print("--------")
start_flan = time.time()
print(f"- Sentiment: ")
conversation_flant5.invoke(input={"text": text_input, "query": sentiment_query});
print("\n- Location: ")
conversation_flant5.invoke(input={"text": text_input, "query": location_query});
print("\n- Time: ")
conversation_flant5.invoke(input={"text": text_input, "query": time_query});
print("\n\n                          ----====----\n")
end_flan = time.time()

print(f"Granite analysis time: {end_granite - start_granite:.2f} seconds")
print(f"Flan analysis time: {end_flan - start_flan:.2f} seconds")

보시다시피 Flan-T5-Large는 파라미터 수가 7억 7천만 개에 불과하기 때문에 일부 결과를 더 빠르게 생성할 수 있습니다.  
하지만 그 결과는 정확도나 세부 묘사 면에서 다소 부족합니다.  
어느 정도까지는 동작하지만, 70억 개의 파라미터를 가진 Granite-7B-Instruct 모델의 결과와는 비교할 수 없습니다.

LLM을 활용하는 데 있어 핵심은, 원하는 성능과 정확도, 그리고 필요한 리소스 및 비용 사이에서 적절한 균형점을 찾는 것입니다.

따라서, 데이터가 변경되거나 모델이 진화하더라도 예상한 동작을 지속적으로 얻기 위해 **신뢰성 점검(Confidence Check)** 을 수행하는 것이 매우 중요합니다.